# Training a Support Vector Machine (SVM) classifier to predict the best crop from the soil and weather conditions.

In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    classification_report, confusion_matrix, log_loss
)


sns.set_theme(style="whitegrid")

In [8]:
df = pd.read_csv("../dataset/crop_recommendation_dataset.csv")

features = ["N", "P", "K", "temperature", "humidity", "ph", "rainfall"]
x = df[features]
y = df["label"]



## 1. Train / Validation / Test Split (60/20/20)

Same split logic as the other models, so that every model is evaluated on identical
data.

In [9]:
x_temp, x_test, y_temp, y_test = train_test_split(
    x, y, test_size=0.20, random_state=42, stratify=y
)

x_train, x_val, y_train, y_val = train_test_split(
    x_temp, y_temp, test_size=0.25, random_state=42, stratify=y_temp
)

print(f"Train: {x_train.shape}")
print(f"Validation: {x_val.shape}")
print(f"Test: {x_val.shape}")

Train: (1320, 7)
Validation: (440, 7)
Test: (440, 7)


## 2. Feature Scaling

SVM is distance-based (it finds the boundary that maximizes the margin between
classes), so features need to be on the same scale.We fit the scaler on the training data only.

In [ ]:
scaler = StandardScaler()
x_train_scaled = scaler.fit_transform(x_train)
x_val_scaled = scaler.transform(x_val)
x_test_scaled = scaler.transform(x_test)

x_train_scaled = pd.DataFrame(x_train_scaled, columns=features, index=x_train.index)
x_val_scaled = pd.DataFrame(x_val_scaled, columns=features, index=x_val.index)
x_test_scaled = pd.DataFrame(x_test_scaled, columns=features, index=x_test.index)



## 3. Choosing C Using the Validation Set
C controls how strictly SVM tries to classify every training point correctly.
Low C allows more tolerance for misclassified points (simpler boundary, less
risk of overfitting). High C tries harder to get every training point right
(more complex boundary, higher risk of overfitting). We test a range of C
values and check accuracy on the validation set.

In [10]:
c_values = [0.01, 0.1, 1, 10, 100]
val_accuracies = []

for c in c_values:
    model = SVC(kernel="rbf", C=c, random_state=42)
    model.fit(x_train_scaled, y_train)
    preds = model.predict(x_val_scaled)
    val_accuracies.append(accuracy_score(y_val, preds))

plt.figure(figsize=(10, 5))
plt.plot(c_values, val_accuracies, marker="o", color="green")
plt.xscale("log")
plt.title("SVM Validation Accuracy for Different Values of C")
plt.xlabel("C (log scale)")
plt.ylabel("Accuracy on Validation Set")
plt.grid(True)
plt.tight_layout()
plt.show()

best_c = c_values[val_accuracies.index(max(val_accuracies))]
print(f"Best C: {best_c}, Validation Accuracy: {max(val_accuracies):.4f}")

NameError: name 'x_train_scaled' is not defined